## Modeling

### 1. Persiapan

#### 1.1. Import Library

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from kneed import KneeLocator
from config import (
  CLEANED_REGENCIES_CSV,
  SCALED_FEATURES_CSV,
  MODEL_PKL,
  CLUSTERED_REGENCIES_CSV,
  SELECTED_FEATURES
)

#### 1.2. Persiapan Data

In [ ]:
df_reg = pd.read_csv(CLEANED_REGENCIES_CSV)
df_scaled = pd.read_csv(SCALED_FEATURES_CSV)

#### 1.3. Parameter

In [ ]:
K_MIN = 2
K_MAX = 10
RANDOM_STATE = 42

### 2. Penentuan Jumlah Klaster Optimal (Elbow Method & Kneedle)

#### 2.1. Perhitungan Within-Cluster Sum of Squares (WCSS)

In [ ]:
scaled_cols = [f"scaled_{c}" for c in SELECTED_FEATURES if f"scaled_{c}" in df_scaled.columns]
X = df_scaled[scaled_cols].values if scaled_cols else df_scaled.values

k_range = list(range(K_MIN, K_MAX + 1))
wcss = [KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto').fit(X).inertia_ for k in k_range]

kn = KneeLocator(k_range, wcss, curve='convex', direction='decreasing')
optimal_k = int(kn.knee) if kn.knee is not None else 4

print(f"Rentang K Diuji     : {K_MIN} hingga {K_MAX}")
print(f"K Optimal Terpilih  : K = {optimal_k}")

#### 2.2. Visualisasi Kurva Elbow

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', color='#1f77b4', linewidth=2, label='Inertia / WCSS')
plt.axvline(x=optimal_k, color='#d62728', linestyle='--', label=f'Optimal K = {optimal_k}')
plt.title('Evaluasi Elbow Method dengan Optimasi Kneedle')
plt.xlabel('Jumlah Klaster (K)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### 3. Pelatihan Model K-Means & Pengelompokan Data

#### 3.1. Fitting Model & Simpan Bobot Model

In [ ]:
final_kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
df_clustered = df_reg.copy()
df_clustered['cluster_label'] = final_kmeans.fit_predict(X)

os.makedirs(os.path.dirname(MODEL_PKL), exist_ok=True)
with open(MODEL_PKL, 'wb') as f:
  pickle.dump(final_kmeans, f)

os.makedirs(os.path.dirname(CLUSTERED_REGENCIES_CSV), exist_ok=True)
df_clustered.to_csv(CLUSTERED_REGENCIES_CSV, index=False)

print(f"Model tersimpan di               : {MODEL_PKL}")
print(f"Dataset terklaster tersimpan di : {CLUSTERED_REGENCIES_CSV}")

#### 3.2. Distribusi Jumlah Anggota per Klaster

In [ ]:
cluster_dist = df_clustered['cluster_label'].value_counts().sort_index()
pd.DataFrame({
  'Klaster': [f"Klaster {c}" for c in cluster_dist.index],
  'Jumlah Kab/Kota': cluster_dist.values,
  'Persentase (%)': (cluster_dist.values / len(df_clustered) * 100).round(2)
})